# Activity 1 Solutions: Cross-Tool Refresher Drills (SQL vs Pandas)

The SQL for each drill is in the activity (run in Snowsight against `W5D1_KICKOFF_CLAIMS`). This notebook holds the matching pandas, executed.

In [1]:
import pandas as pd
import numpy as np

claims = pd.DataFrame({
    "claim_id": ["C-1001", "C-1002", "C-1003", "C-1004", "C-1005", "C-1006",
                 "C-1007", "C-1008", "C-1009", "C-1010", "C-1011", "C-1012"],
    "region": ["Northeast", "Northeast", "Southeast", "Southeast", "Midwest", "Midwest",
               "Northeast", "Southeast", "Midwest", "Northeast", "Southeast", "Midwest"],
    "line": ["Auto", "Home", "Auto", "Home", "Auto", "Home",
             "Auto", "Auto", "Auto", "Home", "Home", "Home"],
    "claim_month": ["2026-01", "2026-01", "2026-01", "2026-02", "2026-02", "2026-02",
                    "2026-03", "2026-03", "2026-03", "2026-04", "2026-04", "2026-04"],
    "paid_amount": [4200, 12500, 3100, 9800, 5200, 15200, 4700, 2900, 6100, 11300, 8600, 13900],
    "open_days": [12, 45, 9, 60, 21, 75, 15, 8, 30, 52, 41, 66],
})

claims

,claim_id,region,line,claim_month,paid_amount,open_days
0,C-1001,Northeast,Auto,2026-01,4200,12
1,C-1002,Northeast,Home,2026-01,12500,45
2,C-1003,Southeast,Auto,2026-01,3100,9
3,C-1004,Southeast,Home,2026-02,9800,60
4,C-1005,Midwest,Auto,2026-02,5200,21
5,C-1006,Midwest,Home,2026-02,15200,75
6,C-1007,Northeast,Auto,2026-03,4700,15
7,C-1008,Southeast,Auto,2026-03,2900,8
8,C-1009,Midwest,Auto,2026-03,6100,30
9,C-1010,Northeast,Home,2026-04,11300,52


### Setup, part 3: the same data in Polars

Convert `claims` to a Polars DataFrame:

In [ ]:
import polars as pl

claims_pl = pl.DataFrame(claims)
claims_pl

## Drill 1: WHERE and ORDER BY

In [2]:
(
    claims[(claims["line"] == "Auto") & (claims["paid_amount"] > 4000)]
    .sort_values("paid_amount", ascending=False)
    [["claim_id", "region", "paid_amount"]]
)

,claim_id,region,paid_amount
8,C-1009,Midwest,6100
4,C-1005,Midwest,5200
6,C-1007,Northeast,4700
0,C-1001,Northeast,4200


### Polars Solution 1

In [ ]:
(
    claims_pl
    .filter((pl.col("line") == "Auto") & (pl.col("paid_amount") > 4000))
    .sort("paid_amount", descending=True)
    .select(["claim_id", "region", "paid_amount"])
)

## Drill 2: GROUP BY

In [3]:
claims.groupby("region")["paid_amount"].agg(
    num_claims="count", total_paid="sum", avg_paid="mean"
)

,num_claims,total_paid,avg_paid
region,,,
Midwest,4,40400,10100.0
Northeast,4,32700,8175.0
Southeast,4,24400,6100.0


### Polars Solution 2

In [ ]:
(
    claims_pl
    .group_by("region")
    .agg(
        pl.len().alias("num_claims"),
        pl.col("paid_amount").sum().alias("total_paid"),
        pl.col("paid_amount").mean().alias("avg_paid")
    )
)

## Drill 3: HAVING

In [4]:
avg_paid = claims.groupby("region")["paid_amount"].mean()
avg_paid[avg_paid > 8000]

region
Midwest      10100.0
Northeast     8175.0
Name: paid_amount, dtype: float64

### Polars Solution 3

In [ ]:
(
    claims_pl
    .group_by("region")
    .agg(pl.col("paid_amount").mean().alias("avg_paid"))
    .filter(pl.col("avg_paid") > 8000)
)

## Drill 4: LAG is shift

In [5]:
monthly = (
    claims.groupby("claim_month", as_index=False)["paid_amount"]
    .sum()
    .rename(columns={"paid_amount": "total_paid"})
)
monthly["prev_total"] = monthly["total_paid"].shift(1)
monthly["change"] = monthly["total_paid"] - monthly["prev_total"]
monthly

,claim_month,total_paid,prev_total,change
0,2026-01,19800,NaN,NaN
1,2026-02,30200,19800.0,10400.0
2,2026-03,13700,30200.0,-16500.0
3,2026-04,33800,13700.0,20100.0


### Polars Solution 4

In [ ]:
(
    monthly_pl
    .with_columns(
        pl.col("total_paid").shift(1).alias("prev_total"),
        (pl.col("total_paid") - pl.col("total_paid").shift(1)).alias("change")
    )
)

## Drill 5: the moving average

Matching the SQL requires `min_periods=1`: the SQL frame averages whatever rows exist so far, so the first month has a value. Default `rolling(3)` would give NaN for the first two months, which does not match the Snowsight grid.

In [6]:
monthly["moving_avg_3m"] = monthly["total_paid"].rolling(3, min_periods=1).mean().round(2)
monthly

,claim_month,total_paid,prev_total,change,moving_avg_3m
0,2026-01,19800,NaN,NaN,19800.00
1,2026-02,30200,19800.0,10400.0,25000.00
2,2026-03,13700,30200.0,-16500.0,21233.33
3,2026-04,33800,13700.0,20100.0,25900.00


### Polars Solution 5

In [ ]:
(
    monthly_pl
    .with_columns(
        pl.col("total_paid").rolling_mean(window_size=3, min_samples=1).round(2).alias("moving_avg_3m")
    )
)

## Drill 6: DENSE_RANK is rank(method="dense")

In [7]:
claims["paid_rank"] = claims["paid_amount"].rank(method="dense", ascending=False).astype(int)
claims.sort_values("paid_rank")[["claim_id", "paid_amount", "paid_rank"]].head(3)

,claim_id,paid_amount,paid_rank
5,C-1006,15200,1
11,C-1012,13900,2
1,C-1002,12500,3


### Polars Solution 6

In [ ]:
(
    claims_pl
    .with_columns(
        pl.col("paid_amount").rank(method="dense", descending=True).alias("paid_rank")
    )
    .sort("paid_rank")
    .head(3)
)

## Drill 7: CASE

In [8]:
claims["severity"] = np.where(
    claims["paid_amount"] >= 10000, "high",
    np.where(claims["paid_amount"] >= 5000, "medium", "low")
)
claims["severity"].value_counts()

severity
low       4
high      4
medium    4
Name: count, dtype: int64

### Polars Solution 7

In [ ]:
(
    claims_pl
    .with_columns(
        pl.when(pl.col("paid_amount") >= 10000).then(pl.lit("high"))
        .when(pl.col("paid_amount") >= 5000).then(pl.lit("medium"))
        .otherwise(pl.lit("low")).alias("severity")
    )
    .group_by("severity")
    .len()
)

`np.select` reads closer to the SQL with many branches:

```python
conditions = [claims["paid_amount"] >= 10000, claims["paid_amount"] >= 5000]
choices = ["high", "medium"]
claims["severity"] = np.select(conditions, choices, default="low")
```

## Wrap up: sample answers

1. Both engines executed the same logic on the same 12 rows; the notation differed, the computation did not.
2. It hands each row the value from the previous row in the declared order; pandas spells it `shift(1)` after an explicit sort.
3. The first two months: SQL's frame averages the rows that exist so far, while default `rolling(3)` refuses to answer until it has 3. `min_periods=1` switches pandas to the SQL contract.
4. `OVER (...)` declares the window: which rows this row may look at, and in what order. Defining that clause precisely is this afternoon's work.